# Metabolite associations and PRS interactions


In [ ]:
# Shared configuration and metabolite analysis inputs.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

try:
    from IPython.display import display
except Exception:
    display = print

SHARED_NOTEBOOK = Path(
    os.environ.get(
        "CHD_SHARED_NOTEBOOK",
        str(Path.cwd() / "00_shared_config.ipynb"),
    )
)

if SHARED_NOTEBOOK.exists():
    get_ipython().run_line_magic("run", f'"{SHARED_NOTEBOOK}"')
elif not os.environ.get("CHD_SENSITIVITY_OUTPUT_DIR") and "SENSITIVITY_OUTPUT_DIR" not in globals():
    raise FileNotFoundError(
        f"Shared notebook not found: {SHARED_NOTEBOOK}. Place this Result 6 notebook beside "
        "00_shared_config.ipynb, set CHD_SHARED_NOTEBOOK, or set "
        "CHD_SENSITIVITY_OUTPUT_DIR to the server analysis root."
    )

server_root = os.environ.get("CHD_SENSITIVITY_OUTPUT_DIR")
if not server_root:
    server_root = globals().get("SENSITIVITY_OUTPUT_DIR")
if not server_root:
    raise RuntimeError("Could not resolve the server analysis root.")

SERVER_SENSITIVITY_OUTPUT_DIR = Path(server_root)
PROJECT_RESULTS_DIR = SERVER_SENSITIVITY_OUTPUT_DIR / "04_results"
RESULT6_ROOT = PROJECT_RESULTS_DIR / "result6_exploratory_metabolites"
RESULTS_OUTPUT_DIR = RESULT6_ROOT
TABLE_DIR = RESULT6_ROOT / "tables"
FIGURE_DIR = RESULT6_ROOT / "figures"
FIGURE_DATA_DIR = RESULT6_ROOT / "source_data"

for directory in [RESULTS_OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, FIGURE_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MAIN_EFFECTS_PATH = Path(
    os.environ.get(
        "CHD_RESULT6_MAIN_EFFECTS_CSV",
        str(TABLE_DIR / "result6_metabolite_main_effects.csv"),
    )
)
INTERACTIONS_PATH = Path(
    os.environ.get(
        "CHD_RESULT6_INTERACTIONS_CSV",
        str(TABLE_DIR / "result6_prs_metabolite_interactions.csv"),
    )
)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Result 6 fits the locked discovery and validation-replication models, then renders figures.")
print("Server root:", SERVER_SENSITIVITY_OUTPUT_DIR)
print("Main effects:", MAIN_EFFECTS_PATH)
print("Interactions:", INTERACTIONS_PATH)


In [ ]:
# Result 6 upstream analysis: 106 prespecified main effects and 106 PRS interactions.
# Execution-only optimizations:
#   1) train-fitted clinical/PRS and NMR preprocessing is performed once;
#   2) immutable NumPy matrices are shared through joblib memmapping;
#   3) each metabolite/family is saved to a signed checkpoint immediately;
#   4) Cox fitting uses lifelines batch_mode and PH diagnostics run in parallel.
# The estimands are unchanged: unpenalized Cox PH with robust SE is primary,
# ridge remains a complete 106+106 effect-stability sensitivity, and the
# discovery/validation populations and BH families remain locked.

from joblib import Parallel, delayed
from lifelines.statistics import proportional_hazard_test
import time

RESULT6_WORKERS = max(1, int(os.environ.get("CHD_RESULT6_WORKERS", "12")))
RESULT6_RIDGE_PENALIZER = float(
    os.environ.get("CHD_RESULT6_RIDGE_PENALIZER", "1e-4")
)
RESULT6_CACHE_CODE_TAG = "result6_precomputed_design_per_metabolite_cache_v3"

df_train, df_val, df_test, final_feature_sets, pipeline_manifest = (
    validate_pipeline_artifacts(load_frames=True)
)

_result6_adjustment = list(final_feature_sets["Clinical_plus_PRS"])
_inverse_nmr = {str(field): name for name, field in NMR_106_FIELD_MAP.items()}
_result6_fields = [str(field) for field in nmr106_cols]

if len(_result6_fields) != 106 or len(set(_result6_fields)) != 106:
    raise RuntimeError("Result 6 requires exactly 106 unique locked NMR fields")
if set(_result6_fields) != set(_inverse_nmr):
    raise RuntimeError("NMR106 field list and NMR_106_FIELD_MAP differ")

def _bh(p_values):
    p = np.asarray(p_values, float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = np.minimum.accumulate(
        (ranked * len(ranked) / np.arange(1, len(ranked) + 1))[::-1]
    )[::-1]
    out = np.empty_like(adjusted)
    out[order] = np.clip(adjusted, 0, 1)
    return out

def _extract_term(model, term):
    row = model.summary.loc[term]
    return {
        "coef": float(row["coef"]),
        "se(coef)": float(row["se(coef)"]),
        "p": float(row["p"]),
        "HR": float(row["exp(coef)"]),
        "HR_CI_Lower": float(row["exp(coef) lower 95%"]),
        "HR_CI_Upper": float(row["exp(coef) upper 95%"]),
    }

# One-time train-fitted preprocessing.
# The base covariates and all 106 NMR variables are processed only once.

log_progress(
    "Result6 PREPROCESS START | clinical+PRS once | locked NMR106 once"
)

(
    _r6_base_train_df,
    _r6_base_val_df,
    _r6_base_test_df,
    _r6_base_columns,
) = prepare_X_train_val_test(
    df_train,
    df_val,
    df_test,
    _result6_adjustment,
    scale=True,
)

(
    _r6_nmr_train_df,
    _r6_nmr_val_df,
    _r6_nmr_test_df,
    _r6_nmr_columns,
) = prepare_X_train_val_test(
    df_train,
    df_val,
    df_test,
    _result6_fields,
    scale=True,
)

_r6_base_columns = list(_r6_base_columns)
_r6_nmr_columns = list(_r6_nmr_columns)

if _r6_nmr_columns != _result6_fields:
    raise RuntimeError(
        "One-time NMR preprocessing changed the locked field order: "
        f"expected={_result6_fields[:5]}..., actual={_r6_nmr_columns[:5]}..."
    )
if PRS_COL not in _r6_base_columns:
    raise RuntimeError(f"PRS column is absent from the processed adjustment matrix: {PRS_COL}")

_r6_train_base = np.ascontiguousarray(
    _r6_base_train_df.to_numpy(dtype=np.float64)
)
_r6_val_base = np.ascontiguousarray(
    _r6_base_val_df.to_numpy(dtype=np.float64)
)
_r6_train_nmr = np.ascontiguousarray(
    _r6_nmr_train_df[_result6_fields].to_numpy(dtype=np.float64)
)
_r6_val_nmr = np.ascontiguousarray(
    _r6_nmr_val_df[_result6_fields].to_numpy(dtype=np.float64)
)

_r6_train_time = np.ascontiguousarray(df_train["Time"].to_numpy(dtype=np.float64))
_r6_train_event = np.ascontiguousarray(df_train["Event"].to_numpy(dtype=np.int8))
_r6_val_time = np.ascontiguousarray(df_val["Time"].to_numpy(dtype=np.float64))
_r6_val_event = np.ascontiguousarray(df_val["Event"].to_numpy(dtype=np.int8))
_r6_field_index = {field: index for index, field in enumerate(_result6_fields)}
_r6_prs_index = _r6_base_columns.index(PRS_COL)

if not all(
    np.isfinite(array).all()
    for array in [_r6_train_base, _r6_val_base, _r6_train_nmr, _r6_val_nmr]
):
    raise RuntimeError("Non-finite values remain after one-time Result 6 preprocessing")

# Fail-closed equivalence audit against the original per-metabolite path.
# This checks three locked fields before any model is fitted.
_r6_equivalence_rows = []
for _audit_field in _result6_fields[:3]:
    _old_train, _old_val, _old_test, _old_columns = prepare_X_train_val_test(
        df_train,
        df_val,
        df_test,
        list(dict.fromkeys(_result6_adjustment + [_audit_field])),
        scale=True,
    )
    _expected_columns = _r6_base_columns + [_audit_field]
    if list(_old_columns) != _expected_columns:
        raise RuntimeError(
            f"Result6 preprocessing equivalence column mismatch for {_audit_field}: "
            f"old={list(_old_columns)}, reused={_expected_columns}"
        )
    _new_train = np.column_stack([
        _r6_train_base,
        _r6_train_nmr[:, _r6_field_index[_audit_field]],
    ])
    _new_val = np.column_stack([
        _r6_val_base,
        _r6_val_nmr[:, _r6_field_index[_audit_field]],
    ])
    _train_difference = float(
        np.max(np.abs(_old_train.to_numpy(dtype=np.float64) - _new_train))
    )
    _val_difference = float(
        np.max(np.abs(_old_val.to_numpy(dtype=np.float64) - _new_val))
    )
    if _train_difference > 1e-10 or _val_difference > 1e-10:
        raise RuntimeError(
            f"Result6 preprocessing reuse is not equivalent for {_audit_field}: "
            f"train_max_abs={_train_difference}, val_max_abs={_val_difference}"
        )
    _r6_equivalence_rows.append({
        "Metabolite": _audit_field,
        "Old_Columns_Equal_Reused": True,
        "Train_Max_Absolute_Difference": _train_difference,
        "Validation_Max_Absolute_Difference": _val_difference,
        "Tolerance": 1e-10,
        "Pass": True,
    })

_r6_equivalence_audit = pd.DataFrame(_r6_equivalence_rows)
atomic_write_csv(
    _r6_equivalence_audit,
    FIGURE_DATA_DIR / "result6_preprocessing_reuse_equivalence_audit.csv",
    index=False,
)

_r6_design_contract = {
    "code_tag": RESULT6_CACHE_CODE_TAG,
    "pipeline_manifest_sha256": current_pipeline_sha256(),
    "train_eid_sha256": eid_digest(df_train),
    "validation_eid_sha256": eid_digest(df_val),
    "train_outcome_sha256": dataframe_digest(
        df_train, ["Eid", "Time", "Event"]
    ),
    "validation_outcome_sha256": dataframe_digest(
        df_val, ["Eid", "Time", "Event"]
    ),
    "base_columns": _r6_base_columns,
    "nmr_columns": _result6_fields,
    "standardization": "training-only prepare_X_train_val_test(scale=True)",
    "interaction": "standardized PRS multiplied by standardized metabolite",
}
_r6_design_contract_sha256 = canonical_json_digest(_r6_design_contract)

RESULT6_CHECKPOINT_DIR = RESULT6_ROOT / "checkpoints_fast_cached_v3"
RESULT6_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

log_progress(
    "Result6 PREPROCESS DONE | "
    f"base_features={len(_r6_base_columns)} | nmr_features=106 | "
    "equivalence_audit=PASS"
)

def _r6_family_key(interaction, penalizer):
    family = "interaction" if interaction else "main"
    penalty = "unpenalized" if float(penalizer) == 0.0 else (
        "ridge_" + safe_filename(f"{float(penalizer):.12g}")
    )
    return f"{family}_{penalty}"

def _r6_checkpoint_payload(field_id, interaction, penalizer):
    return {
        "code_tag": RESULT6_CACHE_CODE_TAG,
        "design_contract_sha256": _r6_design_contract_sha256,
        "pipeline_manifest_sha256": current_pipeline_sha256(),
        "metabolite": str(field_id),
        "interaction": bool(interaction),
        "penalizer": float(penalizer),
        "robust_se": bool(float(penalizer) == 0.0),
        "ties": "efron",
        "batch_mode": True,
    }

def _r6_checkpoint_path(field_id, interaction, penalizer):
    directory = RESULT6_CHECKPOINT_DIR / _r6_family_key(interaction, penalizer)
    directory.mkdir(parents=True, exist_ok=True)
    return directory / f"{safe_filename(field_id)}.json"

def _r6_load_checkpoint(field_id, interaction, penalizer):
    path = _r6_checkpoint_path(field_id, interaction, penalizer)
    if not path.exists():
        return None
    try:
        saved = json.loads(path.read_text(encoding="utf-8"))
        payload = saved.get("signature_payload", {})
        if saved.get("signature") != canonical_json_digest(payload):
            return None
        if payload != _r6_checkpoint_payload(field_id, interaction, penalizer):
            return None
        row = saved.get("row", {})
        if str(row.get("Metabolite")) != str(field_id):
            return None
        required = {
            "coef", "se(coef)", "p", "HR", "HR_CI_Lower", "HR_CI_Upper",
            "Replication_Beta", "Replication_SE", "Replication_P",
            "Replication_HR", "Replication_HR_CI_Lower",
            "Replication_HR_CI_Upper", "Fit_Error", "Replication_Fit_Error",
        }
        if required.difference(row):
            return None
        if str(row.get("Fit_Error", "")).strip() or str(
            row.get("Replication_Fit_Error", "")
        ).strip():
            return None
        return row
    except Exception:
        return None

def _r6_save_checkpoint(row, field_id, interaction, penalizer):
    if str(row.get("Fit_Error", "")).strip() or str(
        row.get("Replication_Fit_Error", "")
    ).strip():
        return
    payload = _r6_checkpoint_payload(field_id, interaction, penalizer)
    atomic_write_json(
        {
            "signature": canonical_json_digest(payload),
            "signature_payload": payload,
            "row": row,
        },
        _r6_checkpoint_path(field_id, interaction, penalizer),
    )

def _r6_fit_frame(split_name, field_id, interaction):
    field_id = str(field_id)
    column_index = _r6_field_index[field_id]
    if split_name == "train":
        base = _r6_train_base
        metabolite = _r6_train_nmr[:, column_index]
        time_values = _r6_train_time
        event_values = _r6_train_event
    elif split_name == "validation":
        base = _r6_val_base
        metabolite = _r6_val_nmr[:, column_index]
        time_values = _r6_val_time
        event_values = _r6_val_event
    else:
        raise ValueError(f"Unknown Result6 split: {split_name}")

    feature_columns = _r6_base_columns + [field_id]
    feature_values = np.column_stack([base, metabolite])
    if interaction:
        interaction_values = base[:, _r6_prs_index] * metabolite
        feature_values = np.column_stack([feature_values, interaction_values])
        feature_columns = feature_columns + ["PRS_x_Metabolite"]

    frame = pd.DataFrame(feature_values, columns=feature_columns)
    frame.insert(0, "Event", event_values)
    frame.insert(0, "Time", time_values)
    return frame

def _fit_one_metabolite(field_id, interaction=False, penalizer=0.0):
    field_id = str(field_id)
    cached = _r6_load_checkpoint(field_id, interaction, penalizer)
    family_key = _r6_family_key(interaction, penalizer)
    if cached is not None:
        log_progress(f"Result6 RESUME {family_key} field={field_id}")
        return cached

    started = time.time()
    name = _inverse_nmr[field_id]
    term = "PRS_x_Metabolite" if interaction else field_id
    base = {
        "Metabolite": field_id,
        "Metabolite_Label": name,
        "Term": term,
        "Estimator": (
            "unpenalized Cox PH with robust SE"
            if float(penalizer) == 0.0
            else f"ridge Cox sensitivity penalizer={float(penalizer):g}"
        ),
        "Discovery_Population": "train",
        "Replication_Population": "validation",
        "Fit_Error": "",
    }

    try:
        train_fit = _r6_fit_frame("train", field_id, interaction)
        model = CoxPHFitter(penalizer=float(penalizer), l1_ratio=0.0)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(
                train_fit,
                duration_col="Time",
                event_col="Event",
                robust=(float(penalizer) == 0.0),
                batch_mode=True,
                show_progress=False,
            )
        base.update(_extract_term(model, term))
    except Exception as error:
        base.update({
            "coef": np.nan,
            "se(coef)": np.nan,
            "p": 1.0 if float(penalizer) == 0.0 else np.nan,
            "HR": np.nan,
            "HR_CI_Lower": np.nan,
            "HR_CI_Upper": np.nan,
            "Fit_Error": repr(error),
        })

    try:
        val_fit = _r6_fit_frame("validation", field_id, interaction)
        validation_model = CoxPHFitter(
            penalizer=float(penalizer),
            l1_ratio=0.0,
        )
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            validation_model.fit(
                val_fit,
                duration_col="Time",
                event_col="Event",
                robust=(float(penalizer) == 0.0),
                batch_mode=True,
                show_progress=False,
            )
        replication = _extract_term(validation_model, term)
        base.update({
            "Replication_Beta": replication["coef"],
            "Replication_SE": replication["se(coef)"],
            "Replication_P": replication["p"],
            "Replication_HR": replication["HR"],
            "Replication_HR_CI_Lower": replication["HR_CI_Lower"],
            "Replication_HR_CI_Upper": replication["HR_CI_Upper"],
            "Replication_Fit_Error": "",
        })
    except Exception as error:
        base.update({
            "Replication_Beta": np.nan,
            "Replication_SE": np.nan,
            "Replication_P": 1.0 if float(penalizer) == 0.0 else np.nan,
            "Replication_HR": np.nan,
            "Replication_HR_CI_Lower": np.nan,
            "Replication_HR_CI_Upper": np.nan,
            "Replication_Fit_Error": repr(error),
        })

    _r6_save_checkpoint(base, field_id, interaction, penalizer)
    elapsed_minutes = (time.time() - started) / 60.0
    status = "OK" if not base["Fit_Error"] and not base["Replication_Fit_Error"] else "ERROR"
    log_progress(
        f"Result6 DONE {family_key} field={field_id} | "
        f"status={status} | elapsed={elapsed_minutes:.1f} min"
    )
    return base

def _run_family(interaction=False, penalizer=0.0):
    label = "interaction" if interaction else "main"
    workers = min(RESULT6_WORKERS, len(_result6_fields))
    log_progress(
        f"Result6 START {label} penalizer={float(penalizer):g} "
        f"fields={len(_result6_fields)} workers={workers} | "
        "preprocessed_design=reused | per_metabolite_cache=enabled"
    )
    started = time.time()
    rows = Parallel(
        n_jobs=workers,
        backend="loky",
        verbose=10,
        max_nbytes="10M",
        mmap_mode="r",
        pre_dispatch=workers,
        batch_size=1,
    )(
        delayed(_fit_one_metabolite)(field, interaction, penalizer)
        for field in _result6_fields
    )
    frame = pd.DataFrame(rows)
    if len(frame) != 106 or set(frame["Metabolite"].astype(str)) != set(
        _result6_fields
    ):
        raise RuntimeError(
            f"Result6 {label} family is not exactly the locked 106 fields"
        )
    if float(penalizer) == 0.0:
        frame["FDR"] = _bh(frame["p"].fillna(1.0))
        frame["Replication_FDR"] = _bh(frame["Replication_P"].fillna(1.0))
        frame["Direction_Replicated"] = (
            np.sign(frame["coef"]) == np.sign(frame["Replication_Beta"])
        )
        frame["Nominal_Replication"] = (
            (frame["Replication_P"] < 0.05)
            & frame["Direction_Replicated"]
        )
    log_progress(
        f"Result6 DONE {label} penalizer={float(penalizer):g} | "
        f"elapsed={(time.time() - started) / 60.0:.1f} min"
    )
    return frame

# Save each complete family immediately so later families cannot erase progress.
main_effects_raw = _run_family(False, 0.0)
atomic_write_csv(main_effects_raw, MAIN_EFFECTS_PATH, index=False)
atomic_write_csv(
    main_effects_raw,
    TABLE_DIR / "result6_metabolite_main_effects_discovery_replication.csv",
    index=False,
)

interactions_raw = _run_family(True, 0.0)
atomic_write_csv(interactions_raw, INTERACTIONS_PATH, index=False)
atomic_write_csv(
    interactions_raw,
    TABLE_DIR / "result6_prs_metabolite_interactions_discovery_replication.csv",
    index=False,
)

main_ridge = _run_family(False, RESULT6_RIDGE_PENALIZER)
atomic_write_csv(
    main_ridge,
    TABLE_DIR / "result6_metabolite_main_effects_ridge_sensitivity.csv",
    index=False,
)

interaction_ridge = _run_family(True, RESULT6_RIDGE_PENALIZER)
atomic_write_csv(
    interaction_ridge,
    TABLE_DIR / "result6_prs_metabolite_interactions_ridge_sensitivity.csv",
    index=False,
)

# Schoenfeld PH diagnostics for the same top 10 primary terms per family.
# The 20 required refits are parallel and use the same preprocessed matrices.
def _r6_ph_one(interaction, field_id):
    field_id = str(field_id)
    term = "PRS_x_Metabolite" if interaction else field_id
    fit = _r6_fit_frame("train", field_id, interaction)
    model = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
    model.fit(
        fit,
        duration_col="Time",
        event_col="Event",
        robust=True,
        batch_mode=True,
        show_progress=False,
    )
    test = proportional_hazard_test(
        model,
        fit,
        time_transform="rank",
    ).summary
    row = test.loc[term]
    result = {
        "Family": "interaction" if interaction else "main",
        "Metabolite": field_id,
        "Metabolite_Label": _inverse_nmr[field_id],
        "Term": term,
        "PH_Test_Statistic": float(row["test_statistic"]),
        "PH_P": float(row["p"]),
    }
    log_progress(
        f"Result6 PH DONE {result['Family']} field={field_id}"
    )
    return result

_r6_ph_tasks = [
    (False, field_id)
    for field_id in main_effects_raw.sort_values("p")["Metabolite"].head(10)
] + [
    (True, field_id)
    for field_id in interactions_raw.sort_values("p")["Metabolite"].head(10)
]

log_progress(
    f"Result6 PH START tasks={len(_r6_ph_tasks)} "
    f"workers={min(RESULT6_WORKERS, len(_r6_ph_tasks))}"
)
ph_rows = Parallel(
    n_jobs=min(RESULT6_WORKERS, len(_r6_ph_tasks)),
    backend="loky",
    verbose=10,
    max_nbytes="10M",
    mmap_mode="r",
    pre_dispatch=min(RESULT6_WORKERS, len(_r6_ph_tasks)),
    batch_size=1,
)(
    delayed(_r6_ph_one)(interaction, field_id)
    for interaction, field_id in _r6_ph_tasks
)
atomic_write_csv(
    pd.DataFrame(ph_rows),
    TABLE_DIR / "result6_top_terms_proportional_hazards_audit.csv",
    index=False,
)
log_progress("Result6 PH DONE")


In [ ]:
# Locked Result 6 specification generated with the upstream results.
RESULT6_SPECIFICATION_ID="result6_unpenalized_robust_discovery_replication_v2"
MODEL_SPECIFICATION={
    "Specification_ID":RESULT6_SPECIFICATION_ID,"Analysis_Population":"Training discovery; validation internal replication","Analysis_Role":"Exploratory metabolite-level association and PRS interaction analysis",
    "Imputation":"Training-only median/most-frequent imputation","Standardization":"Training-only StandardScaler; effects per training-set SD","Metabolite_Scale":"Per 1 training-set SD","PRS_Scale":"Per 1 training-set SD",
    "Main_Effect_Model":"Clinical + PRS + one metabolite","Interaction_Model":"Clinical + PRS + one metabolite + PRS x metabolite","Interaction_Interpretation":"HR is the multiplicative change in metabolite association per 1-SD higher PRS",
    "Adjustment_Set":"final_feature_sets['Clinical_plus_PRS']; no other metabolite and no NMR_score","Estimator":"Unpenalized Cox PH with robust sandwich SE","Penalizer":0.0,
    "Main_Effect_FDR_Family":"Exactly 106 tests, BH including failed fits as p=1","Interaction_FDR_Family":"Exactly 106 tests, BH including failed fits as p=1","Causal_Claim":False,
    "Ridge_Sensitivity":"penalizer=1e-4; effect stability only; penalized-model p-values are not used","PH_Diagnostics":"Top 10 terms per family by discovery p-value","Test_Set_Used":False}
MODEL_SPECIFICATION_PATH=RESULTS_OUTPUT_DIR/"result6_cns_final_model_specification.json"
atomic_write_json(MODEL_SPECIFICATION,MODEL_SPECIFICATION_PATH)
display(pd.DataFrame([MODEL_SPECIFICATION]).T.rename(columns={0:"Value"}))


In [ ]:
# 1. CNS/NPG-style palette, figure export, and explicit color semantics
CNS = {
    "blue": "#3C5488",
    "red": "#E64B35",
    "green": "#00A087",
    "teal": "#4DBBD5",
    "orange": "#F39B7F",
    "purple": "#8491B4",
    "gray": "#7A7A7A",
    "light_gray": "#D9D9D9",
    "black": "#222222",
    "white": "#FFFFFF",
}

FDR_THRESHOLD = 0.05
COLOR_MEANINGS = {
    "Positive coefficient (FDR < 0.05)": CNS["red"],
    "Negative coefficient (FDR < 0.05)": CNS["blue"],
    "Not FDR-significant": CNS["gray"],
}

plt.rcParams.update({
    "figure.dpi": 170,
    "savefig.dpi": 600,
    "font.family": "DejaVu Sans",
    "font.size": 13.0,
    "axes.titlesize": 15.0,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.0,
    "ytick.labelsize": 12.0,
    "legend.fontsize": 11.0,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

FIGURE_RECORDS = []

def clean_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis=grid_axis, color=CNS["light_gray"], linewidth=0.7, alpha=0.75)
    ax.tick_params(length=3.5, width=0.8, color=CNS["black"])

def save_figure(fig, stem):
    saved = []
    for extension in ["png", "pdf", "svg"]:
        path = FIGURE_DIR / f"{stem}.{extension}"
        fig.savefig(path, bbox_inches="tight", facecolor="white")
        saved.append(path)
        FIGURE_RECORDS.append(
            {
                "Artifact_Type": "Figure",
                "Stem": stem,
                "Format": extension,
                "Path": str(path),
                "SHA256": sha256_file(path),
                "Size_Bytes": path.stat().st_size,
            }
        )
    # Explicit display is required because closing the figure suppresses Jupyter output.
    display(fig)
    plt.close(fig)
    print("Saved and displayed:", stem)
    return saved

def direction_legend_handles(include_fdr_line=True, null_label=None):
    handles = [
        Line2D(
            [0], [0], marker="o", linestyle="none", markersize=7,
            markerfacecolor=color, markeredgecolor="white", markeredgewidth=0.6,
            label=meaning,
        )
        for meaning, color in COLOR_MEANINGS.items()
    ]
    if include_fdr_line:
        handles.append(
            Line2D([0], [0], color=CNS["gray"], linestyle="--", linewidth=1.0,
                   label="FDR = 0.05")
        )
    if null_label:
        handles.append(
            Line2D([0], [0], color=CNS["black"], linestyle=":", linewidth=1.0,
                   label=null_label)
        )
    return handles

def add_semantic_legend(ax, include_fdr_line=True, null_label=None, loc="upper left"):
    legend = ax.legend(
        handles=direction_legend_handles(include_fdr_line, null_label),
        title="Color indicates",
        loc=loc,
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        frameon=False,
        handlelength=2.0,
        fontsize=10.5,
        title_fontsize=11.0,
    )
    legend._legend_box.align = "left"
    return legend

color_key = pd.DataFrame(
    {
        "Color_Meaning": list(COLOR_MEANINGS.keys()),
        "Color_Hex": list(COLOR_MEANINGS.values()),
        "FDR_Threshold": FDR_THRESHOLD,
        "Interpretation": [
            "Positive fitted coefficient; association direction only, not causality",
            "Negative fitted coefficient; association direction only, not causality",
            "Saved or recomputed Benjamini-Hochberg FDR is at least 0.05",
        ],
    }
)
COLOR_KEY_PATH = FIGURE_DATA_DIR / "result6_color_legend_key_cns_final.csv"
color_key.to_csv(COLOR_KEY_PATH, index=False)
display(color_key)


In [ ]:
# 2. Canonicalize saved tables and strictly audit HR confidence intervals
def first_column(frame, candidates, required=True):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    if required:
        raise KeyError(
            "None of the required columns were found: " + ", ".join(candidates)
            + ". Available columns: " + ", ".join(map(str, frame.columns))
        )
    return None

def benjamini_hochberg(values):
    p = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    q = np.full(len(p), np.nan, dtype=float)
    valid = np.isfinite(p)
    p_valid = p[valid]
    if len(p_valid) == 0:
        return q
    order = np.argsort(p_valid)
    ranked = p_valid[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0.0, 1.0)
    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    q[valid] = restored
    return q

def color_meaning(beta, fdr):
    if not np.isfinite(beta) or not np.isfinite(fdr) or fdr >= FDR_THRESHOLD:
        return "Not FDR-significant"
    if beta > 0:
        return "Positive coefficient (FDR < 0.05)"
    if beta < 0:
        return "Negative coefficient (FDR < 0.05)"
    return "Not FDR-significant"

def canonicalize_result6(raw, analysis, source_path):
    metabolite_col = first_column(raw, ["Metabolite", "metabolite", "Feature", "Variable"])
    term_col = first_column(raw, ["Term", "term", "Model_Term"], required=False)
    beta_col = first_column(raw, ["coef", "Beta", "beta", "Coefficient"])
    se_col = first_column(raw, ["se(coef)", "SE", "Std_Error", "Standard_Error"], required=False)
    p_col = first_column(raw, ["p", "P", "P_value", "p_value"])
    hr_col = first_column(raw, ["HR", "Hazard_Ratio", "Interaction_HR"])
    lower_col = first_column(
        raw,
        [
            "HR_CI_Lower", "HR_Lower95", "HR_Lower_95", "CI_Lower",
            "Lower95", "HR_lower", "Interaction_HR_Lower95",
        ],
    )
    upper_col = first_column(
        raw,
        [
            "HR_CI_Upper", "HR_Upper95", "HR_Upper_95", "CI_Upper",
            "Upper95", "HR_upper", "Interaction_HR_Upper95",
        ],
    )
    fdr_col = first_column(raw, ["FDR", "q", "Q_value", "FDR_BH"], required=False)
    label_col = first_column(
        raw,
        ["Metabolite_Label", "Readable_Label", "Display_Label", "UKB_Title"],
        required=False,
    )
    error_col = first_column(raw, ["Error", "Fit_Error"], required=False)

    out = pd.DataFrame(index=raw.index)
    out["Source_Row"] = np.arange(len(raw), dtype=int)
    out["Analysis"] = analysis
    out["Metabolite"] = raw[metabolite_col].astype(str)
    out["Term"] = raw[term_col].astype(str) if term_col else out["Metabolite"]
    if label_col:
        label = raw[label_col].astype("string").str.strip()
        label = label.mask(label.isna() | label.eq(""), out["Metabolite"])
        out["Metabolite_Label"] = label.astype(str)
    else:
        out["Metabolite_Label"] = out["Metabolite"]

    duplicate_label = out["Metabolite_Label"].duplicated(keep=False)
    out.loc[duplicate_label, "Metabolite_Label"] = (
        out.loc[duplicate_label, "Metabolite_Label"]
        + " (" + out.loc[duplicate_label, "Metabolite"] + ")"
    )

    out["Beta"] = pd.to_numeric(raw[beta_col], errors="coerce")
    out["SE"] = pd.to_numeric(raw[se_col], errors="coerce") if se_col else np.nan
    out["P"] = pd.to_numeric(raw[p_col], errors="coerce")
    out["HR"] = pd.to_numeric(raw[hr_col], errors="coerce")
    out["HR_Lower95"] = pd.to_numeric(raw[lower_col], errors="coerce")
    out["HR_Upper95"] = pd.to_numeric(raw[upper_col], errors="coerce")
    out["FDR_Recomputed_BH"] = benjamini_hochberg(out["P"])
    out["FDR_Saved"] = pd.to_numeric(raw[fdr_col], errors="coerce") if fdr_col else np.nan
    out["FDR"] = out["FDR_Recomputed_BH"]
    out["FDR_Source"] = "recomputed Benjamini-Hochberg from the complete p-value family (authoritative)"
    out["Fit_Error"] = raw[error_col].fillna("").astype(str) if error_col else ""

    estimable = np.isfinite(out["HR"])
    missing_ci = estimable & (
        ~np.isfinite(out["HR_Lower95"]) | ~np.isfinite(out["HR_Upper95"])
    )
    if missing_ci.any():
        examples = out.loc[missing_ci, "Metabolite"].head(8).tolist()
        raise RuntimeError(
            f"{analysis}: {int(missing_ci.sum())} estimable HR rows lack complete saved 95% CI. "
            f"Examples: {examples}. CI columns selected were {lower_col!r} and {upper_col!r}."
        )

    invalid_ci = estimable & (
        (out["HR"] <= 0)
        | (out["HR_Lower95"] <= 0)
        | (out["HR_Upper95"] <= 0)
        | (out["HR_Lower95"] > out["HR"])
        | (out["HR_Upper95"] < out["HR"])
    )
    if invalid_ci.any():
        examples = out.loc[invalid_ci, "Metabolite"].head(8).tolist()
        raise RuntimeError(f"{analysis}: invalid HR/95% CI ordering for {examples}")

    beta_hr_mismatch = estimable & np.isfinite(out["Beta"]) & (
        np.abs(np.log(out["HR"]) - out["Beta"]) > 1e-5
    )
    if beta_hr_mismatch.any():
        warnings.warn(
            f"{analysis}: {int(beta_hr_mismatch.sum())} rows have log(HR) different from Beta by >1e-5."
        )

    out["Is_FDR_Significant"] = np.isfinite(out["FDR"]) & (out["FDR"] < FDR_THRESHOLD)
    out["Color_Meaning"] = [
        color_meaning(beta, fdr) for beta, fdr in zip(out["Beta"], out["FDR"])
    ]
    out["Color_Hex"] = out["Color_Meaning"].map(COLOR_MEANINGS)
    out["FDR_Threshold"] = FDR_THRESHOLD
    out["MinusLog10FDR"] = -np.log10(out["FDR"].clip(lower=np.finfo(float).tiny))
    out["Source_Path"] = str(source_path)
    out["Source_SHA256"] = sha256_file(source_path)
    out["HR_CI_Source_Columns"] = f"{lower_col} | {upper_col}"
    out["Model_Specification_ID"] = RESULT6_SPECIFICATION_ID
    out["Analysis_Population"] = MODEL_SPECIFICATION["Analysis_Population"]
    out["Imputation"] = MODEL_SPECIFICATION["Imputation"]
    out["Standardization"] = MODEL_SPECIFICATION["Standardization"]
    out["Metabolite_Scale"] = MODEL_SPECIFICATION["Metabolite_Scale"]
    out["PRS_Scale"] = MODEL_SPECIFICATION["PRS_Scale"]
    out["Model_Formula"] = (
        MODEL_SPECIFICATION["Main_Effect_Model"]
        if analysis == "Metabolite main effect"
        else MODEL_SPECIFICATION["Interaction_Model"]
    )
    out["Effect_Interpretation"] = (
        "Adjusted HR per 1-SD metabolite increment"
        if analysis == "Metabolite main effect"
        else MODEL_SPECIFICATION["Interaction_Interpretation"]
    )
    return out


In [ ]:
# 3. Load, canonicalize, audit, and save full figure source data
main_raw = pd.read_csv(MAIN_EFFECTS_PATH)
interaction_raw = pd.read_csv(INTERACTIONS_PATH)

def strict_raw_family_audit(raw, family):
    metabolite_column = first_column(raw, ["Metabolite", "metabolite", "Feature", "Variable"])
    p_column = first_column(raw, ["p", "P", "P_value", "p_value"])
    metabolite_ids = raw[metabolite_column].astype(str)
    p_values = pd.to_numeric(raw[p_column], errors="coerce")
    if len(raw) != 106:
        raise RuntimeError(f"{family}: expected 106 planned rows, found {len(raw)}")
    if metabolite_ids.duplicated().any():
        raise RuntimeError(f"{family}: duplicate metabolite IDs in the BH family")
    if set(metabolite_ids) != set(map(str, nmr106_cols)):
        missing = sorted(set(map(str, nmr106_cols)) - set(metabolite_ids))
        extra = sorted(set(metabolite_ids) - set(map(str, nmr106_cols)))
        raise RuntimeError(f"{family}: metabolite family differs from locked NMR106; missing={missing[:8]}, extra={extra[:8]}")
    if not np.isfinite(p_values.to_numpy(float)).all() or not p_values.between(0.0, 1.0).all():
        raise RuntimeError(f"{family}: discovery P values must be finite and in [0,1] for all 106 BH tests")
    return {
        "Analysis": family, "Planned_Test_N": 106, "Observed_Row_N": int(len(raw)),
        "Unique_Metabolite_N": int(metabolite_ids.nunique()),
        "Finite_Discovery_P_N": int(np.isfinite(p_values.to_numpy(float)).sum()),
        "Exact_Locked_NMR106_Set": True,
    }

strict_family_audit = pd.DataFrame([
    strict_raw_family_audit(main_raw, "Metabolite main effect"),
    strict_raw_family_audit(interaction_raw, "PRS x metabolite interaction"),
])
atomic_write_csv(strict_family_audit, FIGURE_DATA_DIR/"result6_strict_106_plus_106_family_audit.csv", index=False)

main_effects = canonicalize_result6(
    main_raw,
    analysis="Metabolite main effect",
    source_path=MAIN_EFFECTS_PATH,
)
interactions = canonicalize_result6(
    interaction_raw,
    analysis="PRS x metabolite interaction",
    source_path=INTERACTIONS_PATH,
)

MAIN_STANDARDIZED_PATH = FIGURE_DATA_DIR / "result6_metabolite_main_effects_standardized_cns_final.csv"
INTERACTION_STANDARDIZED_PATH = FIGURE_DATA_DIR / "result6_prs_metabolite_interactions_standardized_cns_final.csv"
main_effects.to_csv(MAIN_STANDARDIZED_PATH, index=False)
interactions.to_csv(INTERACTION_STANDARDIZED_PATH, index=False)

def table_audit(frame, source_path):
    saved_vs_bh = np.abs(frame["FDR_Saved"] - frame["FDR_Recomputed_BH"])
    return {
        "Analysis": frame["Analysis"].iloc[0],
        "Rows": len(frame),
        "Planned_Test_N": 106,
        "Unique_Metabolite_N": int(frame["Metabolite"].nunique()),
        "Finite_Discovery_P_N": int(np.isfinite(frame["P"]).sum()),
        "Estimable_HR_Rows": int(np.isfinite(frame["HR"]).sum()),
        "Complete_HR_CI_Rows": int(
            (np.isfinite(frame["HR"]) & np.isfinite(frame["HR_Lower95"]) & np.isfinite(frame["HR_Upper95"])).sum()
        ),
        "FDR_Significant": int(frame["Is_FDR_Significant"].sum()),
        "Positive_FDR_Significant": int(
            (frame["Color_Meaning"] == "Positive coefficient (FDR < 0.05)").sum()
        ),
        "Negative_FDR_Significant": int(
            (frame["Color_Meaning"] == "Negative coefficient (FDR < 0.05)").sum()
        ),
        "Fit_Error_Rows": int(frame["Fit_Error"].str.strip().ne("").sum()),
        "Max_Abs_Saved_vs_Recomputed_BH_FDR": float(saved_vs_bh.max(skipna=True)),
        "HR_CI_Source_Columns": frame["HR_CI_Source_Columns"].iloc[0],
        "Source_Path": str(source_path),
        "Source_SHA256": sha256_file(source_path),
        "Model_Fit_In_This_Notebook": True,
    }

result6_audit = pd.DataFrame(
    [
        table_audit(main_effects, MAIN_EFFECTS_PATH),
        table_audit(interactions, INTERACTIONS_PATH),
    ]
)
AUDIT_PATH = FIGURE_DATA_DIR / "result6_input_ci_fdr_audit_cns_final.csv"
result6_audit.to_csv(AUDIT_PATH, index=False)

print("Canonical Result 6 source tables and CI/FDR audit saved.")
display(result6_audit)
display(main_effects.sort_values("FDR").head(8)[
    ["Metabolite", "Metabolite_Label", "Beta", "HR", "HR_Lower95", "HR_Upper95", "FDR", "Color_Meaning"]
])
display(interactions.sort_values("FDR").head(8)[
    ["Metabolite", "Metabolite_Label", "Beta", "HR", "HR_Lower95", "HR_Upper95", "FDR", "Color_Meaning"]
])


In [ ]:
# 4. Prespecified, direction-neutral selections and per-panel source data
N_FOREST = 10
N_VOLCANO_LABELS_PER_DIRECTION = 4
N_SUPPLEMENTARY_RANKED = 30

def forest_selection(frame, n, purpose):
    significant=frame.loc[frame["Is_FDR_Significant"]].copy()
    pool=significant if not significant.empty else frame.loc[np.isfinite(frame["HR"]) & np.isfinite(frame["HR_Lower95"]) & np.isfinite(frame["HR_Upper95"])].copy()
    if pool.empty: raise RuntimeError(f"No estimable rows are available for {purpose}.")
    selected=pool.sort_values(["FDR","P","Metabolite"]).head(n).copy()
    selected["Selection_Purpose"]=purpose
    selected["Selection_Rule"]=(f"Lowest {n} BH-FDR significant results" if not significant.empty else f"No FDR-significant result; lowest {n} BH-FDR estimates shown descriptively")
    return selected.sort_values(["Beta","FDR"]).reset_index(drop=True)

def balanced_annotation_selection(frame, n_each_direction, purpose):
    significant=frame.loc[frame["Is_FDR_Significant"]].copy()
    pool=significant if not significant.empty else frame.loc[np.isfinite(frame["Beta"])].copy()
    positive=pool.loc[pool["Beta"]>0].sort_values(["FDR","P","Metabolite"]).head(n_each_direction)
    negative=pool.loc[pool["Beta"]<0].sort_values(["FDR","P","Metabolite"]).head(n_each_direction)
    selected=pd.concat([negative,positive],ignore_index=True)
    selected["Selection_Purpose"]=purpose
    selected["Selection_Rule"]=("Lowest BH-FDR significant labels by direction" if not significant.empty else "No FDR-significant result; lowest BH-FDR labels by direction shown descriptively")
    return selected.sort_values(["Beta","FDR"]).reset_index(drop=True)

def ranked_selection(frame, n):
    ranked = frame.sort_values(["FDR", "P", "Metabolite"]).head(n).copy()
    ranked["Selection_Rule"] = f"Lowest {n} BH-FDR values"
    return ranked

main_forest = forest_selection(main_effects, N_FOREST, "Figure 7C forest plot")
interaction_forest = forest_selection(interactions, N_FOREST, "Figure 7D forest plot")
main_volcano_labels = balanced_annotation_selection(main_effects, N_VOLCANO_LABELS_PER_DIRECTION, "Figure 7A annotation labels")
interaction_volcano_labels = balanced_annotation_selection(interactions, N_VOLCANO_LABELS_PER_DIRECTION, "Figure 7B annotation labels")
main_ranked = ranked_selection(main_effects, N_SUPPLEMENTARY_RANKED)
interaction_ranked = ranked_selection(interactions, N_SUPPLEMENTARY_RANKED)

PANEL_SOURCE_PATHS = {
    "Figure 7A": FIGURE_DATA_DIR / "figure7A_metabolite_main_effect_volcano_source_data_cns_final.csv",
    "Figure 7A labels": FIGURE_DATA_DIR / "figure7A_metabolite_main_effect_volcano_labels_cns_final.csv",
    "Figure 7B": FIGURE_DATA_DIR / "figure7B_prs_metabolite_interaction_volcano_source_data_cns_final.csv",
    "Figure 7B labels": FIGURE_DATA_DIR / "figure7B_prs_metabolite_interaction_volcano_labels_cns_final.csv",
    "Figure 7C": FIGURE_DATA_DIR / "figure7C_top_metabolite_main_effect_forest_source_data_cns_final.csv",
    "Figure 7D": FIGURE_DATA_DIR / "figure7D_top_prs_metabolite_interaction_forest_source_data_cns_final.csv",
    "Supplementary 7A": FIGURE_DATA_DIR / "supplementary_result6A_ranked_main_effects_source_data_cns_final.csv",
    "Supplementary 7B": FIGURE_DATA_DIR / "supplementary_result6B_ranked_interactions_source_data_cns_final.csv",
}

main_effects.to_csv(PANEL_SOURCE_PATHS["Figure 7A"], index=False)
main_volcano_labels.to_csv(PANEL_SOURCE_PATHS["Figure 7A labels"], index=False)
interactions.to_csv(PANEL_SOURCE_PATHS["Figure 7B"], index=False)
interaction_volcano_labels.to_csv(PANEL_SOURCE_PATHS["Figure 7B labels"], index=False)
main_forest.to_csv(PANEL_SOURCE_PATHS["Figure 7C"], index=False)
interaction_forest.to_csv(PANEL_SOURCE_PATHS["Figure 7D"], index=False)
main_ranked.to_csv(PANEL_SOURCE_PATHS["Supplementary 7A"], index=False)
interaction_ranked.to_csv(PANEL_SOURCE_PATHS["Supplementary 7B"], index=False)

display(main_forest[["Metabolite_Label", "HR", "HR_Lower95", "HR_Upper95", "FDR", "Color_Meaning"]])
display(interaction_forest[["Metabolite_Label", "HR", "HR_Lower95", "HR_Upper95", "FDR", "Color_Meaning"]])


In [ ]:
# 6. Axis-level plotting functions for separate vector figures
def draw_volcano(ax, frame, labels, title, xlabel, show_legend=True, compact=False):
    draw_order = [
        "Not FDR-significant",
        "Negative coefficient (FDR < 0.05)",
        "Positive coefficient (FDR < 0.05)",
    ]
    for meaning in draw_order:
        subset = frame.loc[frame["Color_Meaning"] == meaning]
        ax.scatter(
            subset["Beta"], subset["MinusLog10FDR"],
            s=20 if compact else (23 if meaning != "Not FDR-significant" else 17),
            color=COLOR_MEANINGS[meaning],
            alpha=0.90 if meaning != "Not FDR-significant" else 0.42,
            edgecolor="white", linewidth=0.35,
            zorder=3 if meaning != "Not FDR-significant" else 2,
        )

    ax.axhline(-np.log10(FDR_THRESHOLD), color=CNS["gray"], linestyle="--", linewidth=1.0)
    ax.axvline(0.0, color=CNS["black"], linestyle=":", linewidth=1.0)

    for index, (_, row) in enumerate(labels.sort_values(["Beta", "FDR"]).iterrows()):
        horizontal = 6 if row["Beta"] >= 0 else -6
        alignment = "left" if row["Beta"] >= 0 else "right"
        vertical = 5 if index % 2 == 0 else -8
        ax.annotate(
            row["Metabolite_Label"],
            xy=(row["Beta"], row["MinusLog10FDR"]),
            xytext=(horizontal, vertical), textcoords="offset points",
            ha=alignment, va="bottom" if vertical > 0 else "top",
            fontsize=10.0, color=CNS["black"],
            arrowprops={"arrowstyle": "-", "color": CNS["light_gray"], "linewidth": 0.55},
        )

    clean_axis(ax, grid_axis="y")
    ax.set_title(title, loc="left", fontweight="bold", pad=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$-\log_{10}$(FDR)")
    if show_legend:
        add_semantic_legend(ax, include_fdr_line=True, null_label=r"Null effect ($\beta=0$)")

def draw_forest(ax, frame, title, xlabel, show_legend=True, compact=False):
    data = frame.sort_values(["Beta", "FDR"]).reset_index(drop=True)
    positions = np.arange(len(data))
    for y, (_, row) in zip(positions, data.iterrows()):
        color = row["Color_Hex"]
        xerr = np.array(
            [[row["HR"] - row["HR_Lower95"]], [row["HR_Upper95"] - row["HR"]]]
        )
        ax.errorbar(
            row["HR"], y, xerr=xerr,
            fmt="o", markersize=5.6 if compact else 6.4,
            color=color, ecolor=color,
            elinewidth=1.35, capsize=2.7, capthick=1.15,
            markeredgecolor="white", markeredgewidth=0.5, zorder=3,
        )

    ax.axvline(1.0, color=CNS["black"], linestyle=":", linewidth=1.0)
    ax.set_xscale("log")
    lower_limit = min(1.0, float(data["HR_Lower95"].min())) * 0.985
    upper_limit = max(1.0, float(data["HR_Upper95"].max())) * 1.015
    ax.set_xlim(lower_limit, upper_limit)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.2f}"))
    ax.set_yticks(positions)
    ax.set_yticklabels(data["Metabolite_Label"], fontsize=10.0 if compact else 11.0)
    ax.set_ylim(-0.7, len(data) - 0.3)
    clean_axis(ax, grid_axis="x")
    ax.set_title(title, loc="left", fontweight="bold", pad=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    if show_legend:
        add_semantic_legend(ax, include_fdr_line=False, null_label="Null effect (HR = 1)")

def draw_ranked_fdr(ax, frame, title, show_legend=True, compact=False):
    data = frame.sort_values(["FDR", "P", "Metabolite"], ascending=[False, False, False]).copy()
    positions = np.arange(len(data))
    ax.barh(
        positions, data["MinusLog10FDR"],
        color=data["Color_Hex"], edgecolor="white", linewidth=0.35, height=0.72,
    )
    ax.axvline(-np.log10(FDR_THRESHOLD), color=CNS["gray"], linestyle="--", linewidth=1.0)
    ax.set_yticks(positions)
    ax.set_yticklabels(data["Metabolite_Label"], fontsize=10.0 if compact else 10.5)
    ax.set_ylim(-0.7, len(data) - 0.3)
    clean_axis(ax, grid_axis="x")
    ax.set_title(title, loc="left", fontweight="bold", pad=8)
    ax.set_xlabel(r"$-\log_{10}$(FDR)")
    ax.set_ylabel("")
    if show_legend:
        add_semantic_legend(ax, include_fdr_line=True, null_label=None)


In [ ]:
# Figure 7A: metabolite main-effect volcano plot
fig, ax = plt.subplots(figsize=(8.6, 6.1))
draw_volcano(
    ax, main_effects, main_volcano_labels,
    "Figure 7A. Metabolite main-effect associations",
    "Log HR per 1-SD metabolite increment",
)
fig.tight_layout()
save_figure(fig, "figure7A_metabolite_main_effect_volcano_cns_final")


In [ ]:
# Figure 7B: PRS-metabolite interaction volcano plot
fig, ax = plt.subplots(figsize=(8.6, 6.1))
draw_volcano(
    ax, interactions, interaction_volcano_labels,
    "Figure 7B. PRS-metabolite interaction signals",
    "Log interaction HR: 1-SD PRS x 1-SD metabolite",
)
fig.tight_layout()
save_figure(fig, "figure7B_prs_metabolite_interaction_volcano_cns_final")


In [ ]:
# Figure 7C: top metabolite main-effect forest plot with complete saved 95% CI
fig, ax = plt.subplots(figsize=(9.0, max(5.1, 0.46 * len(main_forest) + 1.9)))
draw_forest(
    ax, main_forest,
    "Figure 7C. Top metabolite main-effect associations",
    "Adjusted HR per 1-SD metabolite increment (95% CI)",
)
fig.tight_layout()
save_figure(fig, "figure7C_top_metabolite_main_effect_forest_cns_final")


In [ ]:
# Figure 7D: top PRS-metabolite interaction forest plot with complete saved 95% CI
fig, ax = plt.subplots(figsize=(9.0, max(5.1, 0.46 * len(interaction_forest) + 1.9)))
draw_forest(
    ax, interaction_forest,
    "Figure 7D. Top PRS-metabolite interaction signals",
    "Multiplicative interaction HR: 1-SD PRS x 1-SD metabolite (95% CI)",
)
fig.tight_layout()
save_figure(fig, "figure7D_top_prs_metabolite_interaction_forest_cns_final")


In [ ]:
# Supplementary Result 6A: ranked metabolite main effects
fig, ax = plt.subplots(figsize=(9.0, max(6.2, 0.28 * len(main_ranked) + 1.9)))
draw_ranked_fdr(
    ax, main_ranked,
    f"Supplementary Figure 6A. Top {len(main_ranked)} metabolite main-effect signals",
)
fig.tight_layout()
save_figure(fig, "supplementary_result6A_ranked_main_effects_cns_final")


In [ ]:
# Supplementary Result 6B: ranked PRS-metabolite interactions
fig, ax = plt.subplots(figsize=(9.0, max(6.2, 0.28 * len(interaction_ranked) + 1.9)))
draw_ranked_fdr(
    ax, interaction_ranked,
    f"Supplementary Figure 6B. Top {len(interaction_ranked)} PRS-metabolite interaction signals",
)
fig.tight_layout()
save_figure(fig, "supplementary_result6B_ranked_interactions_cns_final")


In [ ]:
# Analysis manifest and reproducibility specification.
specification = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "models_fit_in_this_notebook": True,
    "inline_display": True,
    "combined_overview_figures": False,
    "input_main_effects": str(MAIN_EFFECTS_PATH),
    "input_main_effects_sha256": sha256_file(MAIN_EFFECTS_PATH),
    "input_interactions": str(INTERACTIONS_PATH),
    "input_interactions_sha256": sha256_file(INTERACTIONS_PATH),
    "fdr_threshold": FDR_THRESHOLD,
    "multiple_testing": "Benjamini-Hochberg recomputed from the complete p-value family; saved FDR retained only for audit",
    "forest_selection": f"Lowest {N_FOREST} BH-FDR significant results overall; direction not forced",
    "volcano_annotation_selection": f"Up to {N_VOLCANO_LABELS_PER_DIRECTION} lowest-FDR positive and negative coefficients",
    "supplementary_ranked_n": N_SUPPLEMENTARY_RANKED,
    "color_red": "Positive coefficient with FDR < 0.05",
    "color_blue": "Negative coefficient with FDR < 0.05",
    "color_gray": "Not FDR-significant",
    "interpretive_caution": "Coefficient direction is not a causal claim",
    "confidence_interval_rule": "Use complete saved HR_CI_Lower and HR_CI_Upper (or audited aliases)",
    "scale_rule": MODEL_SPECIFICATION["Standardization"],
    "model_specification_id": RESULT6_SPECIFICATION_ID,
    "analysis_population": MODEL_SPECIFICATION["Analysis_Population"],
    "main_effect_model": MODEL_SPECIFICATION["Main_Effect_Model"],
    "interaction_model": MODEL_SPECIFICATION["Interaction_Model"],
}

SPECIFICATION_PATH = RESULTS_OUTPUT_DIR / "result6_cns_final_plotting_specification.json"
with SPECIFICATION_PATH.open("w", encoding="utf-8") as handle:
    json.dump(specification, handle, indent=2, ensure_ascii=False)

source_artifacts = {
    "Color key": COLOR_KEY_PATH,
    "Input audit": AUDIT_PATH,
    "Standardized main effects": MAIN_STANDARDIZED_PATH,
    "Standardized interactions": INTERACTION_STANDARDIZED_PATH,
    **PANEL_SOURCE_PATHS,
    "Plotting specification": SPECIFICATION_PATH,
    "Reviewed model specification": MODEL_SPECIFICATION_PATH,
}

artifact_rows = list(FIGURE_RECORDS)
for stem, path in source_artifacts.items():
    artifact_rows.append(
        {
            "Artifact_Type": "Source data / audit",
            "Stem": stem,
            "Format": Path(path).suffix.lstrip("."),
            "Path": str(path),
            "SHA256": sha256_file(path),
            "Size_Bytes": Path(path).stat().st_size,
        }
    )

manifest = pd.DataFrame(artifact_rows)
MANIFEST_PATH = RESULTS_OUTPUT_DIR / "result6_cns_final_figure_manifest.csv"
manifest.to_csv(MANIFEST_PATH, index=False)

print("Result 6 final complete.")
print("Manifest:", MANIFEST_PATH)
print("All panels were exported separately, displayed inline, and include their own color legend.")
print("All Result 6 panels were exported as separate figures; no composite was produced.")
display(manifest)


In [ ]:
def _pipeline_context(manifest):
    nested = manifest.get("analysis_context", {}) if isinstance(manifest, dict) else {}
    cohort_id = str(manifest.get("cohort_id", nested.get("cohort_id", "unspecified")))
    analysis_role = str(manifest.get("analysis_role", nested.get("analysis_role", "unspecified")))
    if cohort_id.lower() == "coverage90" and analysis_role.lower() == "primary":
        raise RuntimeError("coverage90 must remain a sensitivity cohort; pipeline manifest labels it primary")
    return cohort_id, analysis_role

cohort_id, analysis_role = _pipeline_context(pipeline_manifest)
locked_ids = set(map(str, nmr106_cols))
for label, frame in [("main", main_effects), ("interaction", interactions)]:
    if len(frame) != 106 or frame["Metabolite"].nunique() != 106 or set(frame["Metabolite"].astype(str)) != locked_ids:
        raise RuntimeError(f"Result 6 {label} family is not the exact locked set of 106 metabolites")
    if not np.isfinite(pd.to_numeric(frame["P"], errors="coerce").to_numpy(float)).all():
        raise RuntimeError(f"Result 6 {label} family has non-finite discovery P values")
context_table = pd.DataFrame([{
    "Result":"Result6", "Cohort_ID":cohort_id, "Analysis_Role":analysis_role,
    "Main_Planned_Test_N":106, "Interaction_Planned_Test_N":106,
    "Primary_Estimator":"unpenalized Cox PH with robust SE", "Ridge_Role":"sensitivity only",
}])
atomic_write_csv(context_table, TABLE_DIR/"result6_analysis_context.csv", index=False)
atomic_write_json({
    "status":"success", "completed_at":str(pd.Timestamp.now()),
    "pipeline_manifest_sha256":current_pipeline_sha256(),
    "cohort_id":cohort_id, "analysis_role":analysis_role,
    "main_family_n":106, "interaction_family_n":106,
    "exact_locked_nmr106_sets":True, "all_discovery_p_finite":True,
    "primary":"unpenalized Cox with robust SE", "ridge_role":"sensitivity only", "test_used":False,
}, RESULT6_ROOT/"SUCCESS.json")
